In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data (do not shuffle or re-sort)
df = pd.read_csv("app_clean.csv")
# Build X
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
].copy()

# Convert Yes/No to 1/0
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Target
y = df["no_show"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Sizes
print("Training size:", len(X_train))
print("Test size:", len(X_test))

# Number of no-shows in test set
test_no_shows = (y_test == 1).sum()
print("No-shows in test set:", test_no_shows)

# Baseline: always predict 'Attended' (no_show = 0)
baseline_accuracy = (y_test == 0).mean()
print("Baseline accuracy:", baseline_accuracy)

Training size: 792
Test size: 198
No-shows in test set: 49
Baseline accuracy: 0.7525252525252525


In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Scale features (fit on training set only)
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

# Predictions
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# Metrics
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", acc)
print("\nConfusion Matrix:")
print(cm)

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)

# Coefficients
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
})

print("\nCoefficients:")
print(coef_df)

# Reminder coefficient
reminder_idx = list(X_train.columns).index("reminder_sent")

scaled_coef = model.coef_[0][reminder_idx]

# Naive odds ratio from scaled feature
or_scaled = np.exp(scaled_coef)

# Odds ratio for changing reminder from 0 -> 1
unscaled_coef = scaled_coef / scaler.scale_[reminder_idx]
or_unscaled = np.exp(unscaled_coef)

print("\nReminder coefficient (scaled):", scaled_coef)
print("exp(coefficient):", or_scaled)

print("\nReminder coefficient adjusted for a 0→1 change:")
print(unscaled_coef)

print("Adjusted odds ratio:")
print(or_unscaled)

Accuracy: 0.7828282828282829

Confusion Matrix:
[[141   8]
 [ 35  14]]

Precision: 0.6363636363636364
Recall: 0.2857142857142857
F1: 0.39436619718309857
ROC-AUC: 0.7809889056293658

Coefficients:
                    feature   coefficient
0                       age -2.451654e-01
1               distance_km  1.358405e-15
2                   fee_inr  0.000000e+00
3                 lead_days  5.852842e-01
4            prior_no_shows  6.245832e-01
5             reminder_sent -5.723416e-01
6       booking_channel_App -3.110035e-02
7     booking_channel_Phone  2.069775e-01
8   booking_channel_Walk-In -1.940596e-01
9   visit_type_Consultation  1.923964e-01
10     visit_type_Follow-up -1.923964e-01

Reminder coefficient (scaled): -0.5723415902934497
exp(coefficient): 0.5642027590128965

Reminder coefficient adjusted for a 0→1 change:
-1.1563034237183694
Adjusted odds ratio:
0.3146471509486568


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ----- Tree with no depth limit -----
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

train_pred_full = tree_full.predict(X_train)
test_pred_full = tree_full.predict(X_test)

train_acc_full = accuracy_score(y_train, train_pred_full)
test_acc_full = accuracy_score(y_test, test_pred_full)

print("Unlimited-depth tree")
print("Training accuracy:", train_acc_full)
print("Test accuracy:", test_acc_full)

# ----- Tree with max_depth = 3 -----
tree_d3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_d3.fit(X_train, y_train)

train_pred_d3 = tree_d3.predict(X_train)
test_pred_d3 = tree_d3.predict(X_test)

train_acc_d3 = accuracy_score(y_train, train_pred_d3)
test_acc_d3 = accuracy_score(y_test, test_pred_d3)

print("\nDepth-3 tree")
print("Training accuracy:", train_acc_d3)
print("Test accuracy:", test_acc_d3)

Unlimited-depth tree
Training accuracy: 0.9949494949494949
Test accuracy: 0.7222222222222222

Depth-3 tree
Training accuracy: 0.8131313131313131
Test accuracy: 0.7676767676767676
